# HW2 Part 1: Data Cleaning & Validation

`ohlc_hw` (raw, left unchanged) → diagnosis → repairs → validated `ohlc_hw_cleaned` in `data/data.db`.

In [1]:
import sqlite3
from pathlib import Path
from contextlib import closing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

%matplotlib inline

## 1.1 Load and inspect

- [ ] Load `ohlc_hw` from `data/data.db`
- [ ] `.info()` and `.describe()`
- [ ] Explain each column (`ts`, `ticker`, raw `open`/`high`/`low`/`close`, vendor `adj_close`, `volume`)
- [ ] Date range and list of tickers

In [2]:
DB_PATH = Path("data/data.db")  # local working copy; never commit it
with closing(sqlite3.connect(DB_PATH)) as conn:
    raw = pd.read_sql("SELECT * FROM ohlc_hw", conn, parse_dates=["ts"])


print("Database Info")
raw.info()

print("\nDatabase Description")
display(raw.describe())

print("\nList of tickers: ", raw['ticker'].unique().tolist())
print("\nDate Range: from ", raw['ts'].min(), "to", raw['ts'].max())

Database Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   ts         397711 non-null  datetime64[ns]
 1   open       397711 non-null  float64       
 2   high       397711 non-null  float64       
 3   low        397711 non-null  float64       
 4   close      397711 non-null  float64       
 5   adj_close  377856 non-null  float64       
 6   volume     397711 non-null  int64         
 7   ticker     397711 non-null  object        
dtypes: datetime64[ns](1), float64(5), int64(1), object(1)
memory usage: 24.3+ MB

Database Description


,ts,open,high,low,close,adj_close,volume
count,397711,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,1998-04-17 21:18:45.292485888,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
min,1962-01-02 00:00:00,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,1985-05-28 00:00:00,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,2000-01-06 00:00:00,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,2012-08-21 00:00:00,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,2024-12-06 00:00:00,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09
std,NaN,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08



List of tickers:  ['CAT', 'CVX', 'HD', 'AAPL', 'TRV', 'JNJ', 'DIS', '<a target="_blank"><span style="color: blue">CSCO</span></a>', 'SHW', 'KO', '<a target="_blank"><span style="font-weight: bold; color: blue">HD</span></a>', '<a target="_blank"><span style="font-weight: bold; color: blue">VZ</span></a>', 'IBM', 'PG', 'HON', 'AXP', 'MMM', 'GS', 'BA', 'CRM', 'UNH', 'NKE', 'NVDA', 'AMGN', 'CSCO', 'MCD', '<a target="_blank"><span style="color: red">AMGN</span></a>', 'VZ', 'V', '<a target="_blank"><span style="color: red">DIS</span></a>', '<a target="_blank"><span style="font-weight: bold; color: blue">NKE</span></a>', '<a target="_blank"><span style="color: blue">TRV</span></a>', 'MRK', 'WMT', '<a target="_blank"><span style="font-weight: bold; color: red">SHW</span></a>', '<a target="_blank"><span style="font-weight: bold; color: red">JNJ</span></a>', '<a target="_blank"><span style="font-weight: bold; color: red">PG</span></a>', '<a target="_blank"><span style="font-weight: bold; color

**Columns**

| Column | Meaning |
|---|---|
| `ts` | Trading date (observation date) |
| `ticker` | Stock symbol; the raw values contain HTML markup that must be normalized |
| `open`, `high`, `low`, `close` | Vendor daily prices, already split-adjusted (not dividend-adjusted) |
| `adj_close` | Vendor close adjusted for splits **and** dividends; used for report returns |
| `volume` | Shares traded that day |

## 1.2 Detect data quality issues

### Ticker labels and duplicate records

- [ ] Normalize ticker markup, whitespace and case **before** any grouping or deduplication; keep the original label or an audit
- [ ] Remove exact duplicates; investigate conflicting `(ticker, ts)` keys instead of keeping the first row

In [3]:
with closing(sqlite3.connect(DB_PATH)) as conn:
    ref = pd.read_sql("SELECT * FROM ohlc", conn, parse_dates=["ts"])

df = raw.copy()
df["raw_ticker"] = df["ticker"]
df["ticker"] = (df["ticker"].str.replace(r"<[^>]+>", "", regex=True).str.strip().str.upper())
print("Tickers before normalization:", df["raw_ticker"].nunique())
print("Tickers after normalization:", df["ticker"].nunique())

exact = df.drop(columns="raw_ticker").duplicated()
print("Exact duplicate rows:", exact.sum())
df = df[~exact].sort_values(["ticker", "ts"]).reset_index(drop=True)

conflicts = df[df.duplicated(["ticker", "ts"], keep=False)]
print("Rows with conflicting (ticker, ts) keys:", len(conflicts))

print(sorted(df["ticker"].unique()))

Tickers before normalization: 150
Tickers after normalization: 30
Exact duplicate rows: 29460
Rows with conflicting (ticker, ts) keys: 0
['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']


### Missing data

- [ ] Missing values per column (including `adj_close`)
- [ ] Missing trading days per ticker against a stated reference calendar (e.g. that ticker's dates in `ohlc`)
- [ ] Summary: "Ticker X is missing Y trading days between [start] and [end]"; report zero honestly
- [ ] Demonstrate the gap detector on a tiny example with one deliberately removed date

In [4]:
print("Missing Cells:", df.isna().sum())
print("Non-finite prices:", np.isinf(df[["open", "high", "low", "close", "adj_close"]]).sum().sum())

gaps = []
for t in df['ticker'].unique():
    actual = pd.DatetimeIndex(df.loc[df['ticker'] == t, 'ts'])
    expected = pd.DatetimeIndex(ref.loc[ref['ticker'] == t, 'ts'])
    missing = expected.difference(actual)
    print(f"Ticker {t} is missing {len(missing)} trading days between {expected.min().date()} and {expected.max().date()}")
    gaps += [{"ticker": t, "ts": d} for d in missing]

print(gaps)

Missing Cells: ts                0
open              0
high              0
low               0
close             0
adj_close     18374
volume            0
ticker            0
raw_ticker        0
dtype: int64
Non-finite prices: 0
Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and

Reference calendar: each ticker's own dates in the supplied `ohlc` table, over that ticker's history. The real data has 0 missing trading days. The demo below removes one date on purpose to show the detector works.

In [5]:
demo_expected = pd.DatetimeIndex(["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"])
demo_actual = demo_expected.drop(pd.Timestamp("2024-01-03"))
demo_missing = demo_expected.difference(demo_actual)
print("Example detector found:", demo_missing.date.tolist())
assert demo_missing.tolist() == [pd.Timestamp("2024-01-03")]

Example detector found: [datetime.date(2024, 1, 3)]


### Outliers

- [ ] Simple daily returns `(close - prev_close) / prev_close`, sorted and within ticker
- [ ] Flag `abs(return - mean) > 3 * std` (sample std, `ddof=1`), both tails, within ticker; handle undefined returns and zero/undefined std
- [ ] Flag `high < close` or `low > close`
- [ ] Flag nonpositive or non-finite prices, including `adj_close`

In [6]:
df["ret"] = df.groupby("ticker")["close"].pct_change(fill_method=None)
mean = df.groupby("ticker")["ret"].transform("mean")
std = df.groupby("ticker")["ret"].transform("std")
df["3sig_tail"] = (df["ret"] - mean).abs().gt(3 * std) & std.gt(0)
df.head()

,ts,open,high,low,close,adj_close,volume,ticker,raw_ticker,ret,3sig_tail
0,1980-12-12,0.12835,0.12835,0.12835,0.12835,0.098941,469034068,AAPL,AAPL,NaN,False
1,1980-12-15,0.12165,0.12165,0.12165,0.12165,0.093779,175884971,AAPL,AAPL,-0.052201,False
2,1980-12-16,0.11272,0.11272,0.11272,0.11272,0.086896,105728103,AAPL,AAPL,-0.073407,False
3,1980-12-17,0.11551,0.11551,0.11551,0.11551,0.089046,86441684,AAPL,AAPL,0.024752,False
4,1980-12-18,0.11886,0.11886,0.11886,0.11886,0.091627,73449671,AAPL,AAPL,0.029002,False


### Inconsistencies

- [ ] `low <= open <= high` and `low <= close <= high`
- [ ] Zero-volume days

In [7]:
P = ["open", "high", "low", "close"]

def add_price_flags(frame):
    """Add one boolean column per price/volume rule."""
    frame = frame.copy()
    frame["high_lt_close"] = frame["high"].lt(frame["close"])
    frame["low_gt_close"] = frame["low"].gt(frame["close"])
    frame["ohlc_bounds"] = ~(frame["low"].le(frame["open"]) & frame["open"].le(frame["high"])
                             & frame["low"].le(frame["close"]) & frame["close"].le(frame["high"]))
    frame["nonpositive_price"] = frame[P + ["adj_close"]].le(0).any(axis=1)
    frame["nonfinite_price"] = np.isinf(frame[P + ["adj_close"]]).any(axis=1)
    frame["zero_volume"] = frame["volume"].eq(0)
    return frame

df = add_price_flags(df)
flag_cols = ["3sig_tail", "high_lt_close", "low_gt_close", "ohlc_bounds",
             "nonpositive_price", "nonfinite_price", "zero_volume"]
print(df[flag_cols].sum())

# Injected example: cases the real data doesn't contain (high < close, low > close, infinite price)
demo = pd.DataFrame({"open": [10, 10, 10], "high": [11, 9, 11], "low": [9, 9, 10.5],
                     "close": [10, 10, 10], "adj_close": [10, 10, np.inf], "volume": [100, 100, 100]})
demo_flags = add_price_flags(demo)
display(demo_flags[["high_lt_close", "low_gt_close", "ohlc_bounds", "nonfinite_price"]])
assert demo_flags["high_lt_close"].tolist() == [False, True, False]
assert demo_flags["low_gt_close"].tolist() == [False, False, True]
assert demo_flags["nonfinite_price"].tolist() == [False, False, True]

3sig_tail            5056
high_lt_close           0
low_gt_close            0
ohlc_bounds             5
nonpositive_price       5
nonfinite_price         0
zero_volume            75
dtype: int64


,high_lt_close,low_gt_close,ohlc_bounds,nonfinite_price
0,False,False,False,False
1,True,False,True,False
2,False,True,True,True


In [8]:
ref_factor = ref.assign(ref_factor=ref["adj_close"] / ref["close"])[["ticker", "ts", "ref_factor"]]
df = df.merge(ref_factor, on=["ticker", "ts"], how="left")
df["adj_missing"] = df["adj_close"].isna()
df["adj_mismatch"] = df["adj_close"].notna() & ~np.isclose(df["adj_close"] / df["close"], df["ref_factor"])
print(df[["adj_missing", "adj_mismatch"]].sum())

adj_missing     18374
adj_mismatch    10417
dtype: int64


### Data Quality Report

Columns: `ticker | issue_type | issue_date | description | severity`

- [ ] Multiple issues per observation allowed; missing dates as separate rows
- [ ] Keep this pre-repair diagnosis; rerun validation after repairs
- [ ] Small injected examples for required cases absent from the real data

In [9]:
checks = [
    ("adj_missing", "missing_adj_close", "adj_close is missing", "high"),
    ("adj_mismatch", "bad_adj_close", "adj_close/close disagrees with the ohlc reference factor", "high"),
    ("nonpositive_price", "nonpositive_price", "a price is zero or negative", "high"),
    ("nonfinite_price", "nonfinite_price", "a price is infinite", "high"),
    ("high_lt_close", "high_below_close", "high < close", "high"),
    ("low_gt_close", "low_above_close", "low > close", "high"),
    ("ohlc_bounds", "ohlc_bounds", "open or close outside [low, high]", "high"),
    ("3sig_tail", "extreme_return", "daily return more than 3 std from the ticker's mean", "medium"),
    ("zero_volume", "zero_volume", "volume is 0 (possible stale data)", "low"),
]
parts = [df.loc[df[col], ["ticker", "ts"]].assign(issue_type=issue, description=desc, severity=sev)
         for col, issue, desc, sev in checks]
parts.append(pd.DataFrame(gaps, columns=["ticker", "ts"]).assign(
    issue_type="missing_trading_day", description="date in the ohlc reference but absent", severity="medium"))

quality_report = (pd.concat(parts, ignore_index=True)
                    .rename(columns={"ts": "issue_date"})
                    [["ticker", "issue_type", "issue_date", "description", "severity"]])
display(quality_report.groupby(["issue_type", "severity"]).size())
quality_report.head()

/var/folders/vl/xcbt_9650nz8tknq733z87640000gn/T/ipykernel_27773/2735558362.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  quality_report = (pd.concat(parts, ignore_index=True)


issue_type         severity
bad_adj_close      high        10417
extreme_return     medium       5056
missing_adj_close  high        18374
nonpositive_price  high            5
ohlc_bounds        high            5
zero_volume        low            75
dtype: int64

,ticker,issue_type,issue_date,description,severity
0,AAPL,missing_adj_close,1981-01-08,adj_close is missing,high
1,AAPL,missing_adj_close,1981-01-13,adj_close is missing,high
2,AAPL,missing_adj_close,1981-02-12,adj_close is missing,high
3,AAPL,missing_adj_close,1981-04-16,adj_close is missing,high
4,AAPL,missing_adj_close,1981-05-19,adj_close is missing,high


**Severity choices**

- **High:** the price itself is wrong or unusable and would corrupt returns and risk metrics if left in (missing or inconsistent `adj_close`, nonpositive or non-finite prices, OHLC rule violations).
- **Medium:** needs review but may be genuine (extreme returns, missing trading days).
- **Low:** informational; prices are unaffected (zero volume, all in 1962–1982 records).

## 1.3 Clean the data

### Missing values

- [ ] Drop rows where **all** of `open`/`high`/`low`/`close` are missing, before filling
- [ ] Forward-fill missing prices within each ticker (sorted by date); recheck OHLC bounds afterwards
- [ ] Missing volume → 0 plus a repair flag
- [ ] Missing `adj_close`: consistent adjustment convention (e.g. date-matched factor from `ohlc`); document the source; quarantine unresolved rows

In [10]:
def repair_missing_values(frame):
    frame = frame[~frame[P].isna().all(axis=1)].copy()
    frame["price_filled"] = frame[P].isna().any(axis=1)
    frame[P] = frame.groupby("ticker")[P].ffill()   # within ticker only; frame is sorted by ticker, ts
    frame["volume_repaired"] = frame["volume"].isna()  # unknown volume is not "no trading", so flag it
    frame["volume"] = frame["volume"].fillna(0)
    return frame

n_before = len(df)
df = repair_missing_values(df)
print("Rows with all OHLC missing (dropped):", n_before - len(df))
print("Raw prices forward-filled:", df["price_filled"].sum())
print("Volume values filled with 0:", df["volume_repaired"].sum())

# Injected example: the real data has none of these cases
demo_repair = pd.DataFrame({"ticker": ["X", "X", "X", "Y"],
                             "open": [10, np.nan, np.nan, np.nan], "high": [11, np.nan, 12, np.nan],
                             "low": [9, np.nan, 10, np.nan], "close": [10, np.nan, 11, np.nan],
                             "volume": [100, 200, np.nan, 50]})
display(repair_missing_values(demo_repair))  # row 2 dropped, row 3 filled from row 1, row 4 dropped, volume flagged

# adj_close: rebuild missing or inconsistent values as close x the ohlc factor for the same (ticker, ts)
df["adj_close_repaired"] = df["adj_missing"] | df["adj_mismatch"]
df.loc[df["adj_close_repaired"], "adj_close"] = df["close"] * df["ref_factor"]
print("adj_close rebuilt from the ohlc factor:", df["adj_close_repaired"].sum(),
      "| still missing:", df["adj_close"].isna().sum())

Rows with all OHLC missing (dropped): 0
Raw prices forward-filled: 0
Volume values filled with 0: 0


,ticker,open,high,low,close,volume,price_filled,volume_repaired
0,X,10.0,11.0,9.0,10.0,100.0,False,False
2,X,10.0,12.0,10.0,11.0,0.0,True,True


adj_close rebuilt from the ohlc factor: 28791 | still missing: 0


### Outliers and impossible prices

- [ ] Impossible OHLC relationships: pick option 1 / 2 / 3 and document it
- [ ] Extreme returns: investigate (split? data error? news?); interpolate only confirmed errors between valid neighbours, otherwise keep and flag or quarantine
- [ ] Recheck raw bounds and adjusted-price consistency after every repair

In [11]:
bad = df["nonpositive_price"] | df["ohlc_bounds"]
quarantine = df[bad].assign(reason="open = 0 on the ticker's first day; no earlier price to repair from")
display(quarantine[["ticker", "ts", "open", "high", "low", "close", "reason"]])
df = df[~bad]

check = df[["ticker", "ts"] + P].merge(ref[["ticker", "ts"] + P], on=["ticker", "ts"], suffixes=("", "_ref"))
print("Raw OHLC values that differ from the ohlc reference:",
      sum(int((~np.isclose(check[c], check[c + "_ref"])).sum()) for c in P))
print("Extreme returns kept and flagged for review:", df["3sig_tail"].sum())

,ticker,ts,open,high,low,close,reason
87103,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000,open = 0 on the ticker's first day; no earlier...
167733,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338,open = 0 on the ticker's first day; no earlier...
213412,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955,open = 0 on the ticker's first day; no earlier...
243893,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537,open = 0 on the ticker's first day; no earlier...
287040,PG,1962-01-02,0.0,1.44000,1.42000,1.43000,open = 0 on the ticker's first day; no earlier...


Raw OHLC values that differ from the ohlc reference: 0
Extreme returns kept and flagged for review: 5056


**Repair decisions**

- Impossible prices (5 rows, `open = 0`): quarantined. Each is the first day of that ticker's history (CVX, JNJ, MRK, PG on 1962-01-02; MCD on 1966-07-05), so options 1 and 2 have no earlier price to use, and option 3 would invent an open. Removing them does not affect any recent analysis window.
- Extreme returns (>3σ): kept and flagged in `3sig_tail`. The raw OHLC matches the supplied `ohlc` reference exactly, so these are consistent vendor prices (real market moves such as 1987, 2008 and 2020), not entry errors. Nothing is interpolated because no error is confirmed.
- Zero volume (75 rows, 1962–1982): kept and flagged; likely sparse records in old data rather than wrong prices.
- `adj_close` (18,374 missing + 10,417 inconsistent): rebuilt as `close × (ohlc.adj_close / ohlc.close)` for the same `(ticker, ts)`, flagged in `adj_close_repaired`. The factor is date-specific, so it is never carried across a corporate action.

### Validation and save

- [ ] No missing, non-finite or nonpositive values in retained raw and adjusted prices
- [ ] Unique `(ticker, ts)`; finite, nonnegative volume; all price relationships valid
- [ ] Before/after statistics
- [ ] Account for retained, removed and quarantined rows
- [ ] Save `ohlc_hw_cleaned` with repair flags and `ts` as ISO `YYYY-MM-DD`; read it back to confirm

In [12]:
clean = add_price_flags(df)
assert not clean[["high_lt_close", "low_gt_close", "ohlc_bounds", "nonpositive_price", "nonfinite_price"]].any().any()
assert clean[P + ["adj_close"]].notna().all().all()
assert not clean.duplicated(["ticker", "ts"]).any()
assert clean["volume"].ge(0).all()
assert np.allclose(clean["adj_close"] / clean["close"], clean["ref_factor"])

print(f"Raw rows {len(raw):,} - exact duplicates {exact.sum():,} - quarantined {len(quarantine)} = saved {len(clean):,}")
display(pd.concat({"before (raw)": raw[P + ["adj_close", "volume"]].describe().T,
                   "after (clean)": clean[P + ["adj_close", "volume"]].describe().T}, axis=1))

out = clean[["ts", "ticker", "open", "high", "low", "close", "adj_close", "volume", "raw_ticker",
             "adj_close_repaired", "volume_repaired", "price_filled", "3sig_tail", "zero_volume"]].copy()

out["ts"] = out["ts"].dt.strftime("%Y-%m-%d")

with closing(sqlite3.connect(DB_PATH)) as conn:
    out.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)
    back = pd.read_sql("SELECT COUNT(*) AS n, MIN(ts) AS first, MAX(ts) AS last FROM ohlc_hw_cleaned", conn)
display(back)
assert back["n"].iloc[0] == len(out)

Raw rows 397,711 - exact duplicates 29,460 - quarantined 5 = saved 368,246


before (raw)                                                      \
                 count          mean           std       min           25%   
open          397711.0  4.628548e+01  6.713259e+01  0.000000  4.209938e+00   
high          397711.0  4.674766e+01  6.777821e+01  0.005208  4.240000e+00   
low           397711.0  4.581628e+01  6.647718e+01  0.004801  4.160000e+00   
close         397711.0  4.629290e+01  6.714188e+01  0.005208  4.200000e+00   
adj_close     377856.0  5.165225e+01  1.945840e+02  0.002945  1.390000e+00   
volume        397711.0  3.097773e+07  1.266237e+08  0.000000  1.730700e+06   

                                                 after (clean)                \
                  50%          75%           max         count          mean   
open            19.85        57.53  6.042600e+02      368246.0  4.630939e+01   
high            20.06        58.13  6.086300e+02      368246.0  4.677182e+01   
low             19.63        56.95  5.978800e+02      368246.0  4.584038e+01   
close           19.86        57.53  6.054000e+02      368246.0  4.631733e+01   
adj_close       11.96        43.77  1.044251e+04      368246.0  3.727467e+01   
volume     4648200.00  11023796.50  9.276606e+09      368246.0  3.095184e+07   

                                                                         \
                    std       min         25%         50%           75%   
open       6.714325e+01  0.005208        4.21       19.89  5.756000e+01   
high       6.778932e+01  0.005208        4.25       20.09  5.816000e+01   
low        6.648945e+01  0.004801        4.17       19.67  5.699750e+01   
close      6.715494e+01  0.005208        4.21       19.91  5.757000e+01   
adj_close  6.358787e+01  0.002945        1.31       11.46  4.157000e+01   
volume     1.266153e+08  0.000000  1731514.50  4649648.00  1.102160e+07   

                         
                    max  
open       6.042600e+02  
high       6.086300e+02  
low        5.978800e+02  
close      6.054000e+02  
adj_close  6.054000e+02  
volume     9.276606e+09

,n,first,last
0,368246,1962-01-02,2024-12-06


## Summary of cleaning decisions

| Finding | Count | Action |
|---|---|---|
| Ticker labels with HTML markup / mixed variants | 150 raw → 30 tickers | Stripped tags, whitespace, upper-cased; kept `raw_ticker` |
| Exact duplicate rows | 29,460 | Removed |
| Conflicting `(ticker, ts)` keys | 0 | None needed |
| Missing trading days vs `ohlc` | 0 | None needed (detector demonstrated on an injected gap) |
| Missing `adj_close` | 18,374 | Rebuilt from the `ohlc` factor, flagged |
| Inconsistent `adj_close` (disagrees with the `ohlc` factor) | 10,417 | Rebuilt from the `ohlc` factor, flagged |
| `open = 0` on first day of history | 5 | Quarantined with reason |
| Returns beyond 3σ | 5,056 | Kept, flagged for review |
| Zero volume | 75 | Kept, flagged |

397,711 raw rows − 29,460 duplicates − 5 quarantined = **368,246 rows** saved to `ohlc_hw_cleaned`. The raw `ohlc_hw` table is unchanged.